# E1 v3 reference runner - B0 and random-k

Use this notebook for the heavy reference shards. Session 3 and Session 4 should use different `REPEATS` and `SHARD_NAME`; their outputs never overwrite each other.

For two sessions, keep the complete k grid in both shards: Session 3 = repeats `[0, 1]`, all k; Session 4 = repeats `[2, 3, 4]`, all k. This covers every required `(repeat, fold, k, draw)` exactly once.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')

# Chạy GSE10245 trước. Đổi COHORT sau khi cả Session 3 và 4 của cohort này đã được lưu thành Kaggle Dataset.
COHORT = 'gse10245'
SESSION = 3  # 3: B0 + random-k nhỏ; 4: random-k lớn
SESSION_PROFILES = {
    3: {'suffix': 'k_small', 'k_grid': [1, 2, 3, 4], 'run_b0': True},
    4: {'suffix': 'k_large', 'k_grid': [5, 10, 25, 50], 'run_b0': False},
}
profile = SESSION_PROFILES[SESSION]
SHARD_NAME = f'{COHORT}_session_{SESSION}_{profile["suffix"]}'
REPEATS = [0, 1, 2, 3, 4]  # mỗi shard luôn đủ 5x5; chỉ chia theo k
K_GRID = profile['k_grid']
RUN_B0 = profile['run_b0']
N_RANDOM_DRAWS = 100
assert COHORT in ['tcga_luad_lusc_primary_tumor', 'gse10245', 'gse30219', 'gse37745', 'gse42127']
assert set(REPEATS) == {0, 1, 2, 3, 4}
print({'cohort': COHORT, 'session': SESSION, 'shard': SHARD_NAME, 'k_grid': K_GRID, 'run_b0': RUN_B0})


In [ ]:
# Keep Kaggle's own compatible numerical stack. requirements.txt pins the local
# workstation stack and must not be installed into Kaggle's shared Python image.
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
RUNTIME_IMPORTS = ('numpy', 'pandas', 'scipy', 'sklearn', 'tables', 'joblib', 'threadpoolctl')
try:
    for package in RUNTIME_IMPORTS:
        print(package, __import__(package).__version__)
except Exception as exc:
    raise RuntimeError('Kaggle numerical runtime is not coherent. Restart the Kaggle session, then rerun from Cell 1; do not run pip install -r requirements.txt.') from exc
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    source, target = INPUT_V3_DATASET / folder, WORK_ROOT / folder
    assert source.is_dir(), f'Missing Kaggle input: {source}'
    if not target.exists():
        target.symlink_to(source, target_is_directory=True)
os.environ['KLTN_DATA_ROOT'] = str(WORK_ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('commit:', COMMIT)


In [ ]:
# Evaluate B0 and random-k only. Random panels are deterministic from cohort, split seed, fold, k and draw.
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from survey_v2 import config as C
from survey_v2.classify import evaluate_panel
from survey_v2.data import load_dataset, variance_prefilter, sha256
from survey_v2.e1_splits import validate_repeated_split_manifest
from scripts.run_e1_feature_selection import split_indices

def atomic_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

def checkpoint(root, label):
    return root / 'checkpoints' / label

ds = load_dataset(WORK_ROOT, COHORT)
split_file = ds.data_dir / 'repeated_splits.csv'
manifest = pd.read_csv(split_file)
errors = validate_repeated_split_manifest(manifest, n_folds=C.N_FOLDS)
assert not errors, errors
output_root = WORK_ROOT / 'results' / 'survey_v3' / 'E1_references' / COHORT / SHARD_NAME

for repeat in REPEATS:
    for fold in range(C.N_FOLDS):
        train, test = split_indices(ds, manifest, repeat, fold)
        seed = int(manifest[(manifest.repeat == repeat) & (manifest.fold == fold)].seed.iloc[0])
        if RUN_B0:
            path = checkpoint(output_root, f'b0_r{repeat}_f{fold}').with_suffix('.csv')
            if not path.is_file():
                with threadpool_limits(1):
                    result = evaluate_panel(ds.X[train], ds.y[train], ds.X[test], ds.y[test], classifier='lr_l2', seed=C.seed_for(C.dataset_seed_id(COHORT), seed, fold, 0), n_jobs=1)
                atomic_csv(pd.DataFrame([{'cohort': COHORT, 'repeat': repeat, 'fold': fold, 'method': 'b0', 'k': ds.X.shape[1], 'classifier': 'lr_l2', 'auroc': result.auroc, 'bacc': result.bacc, 'mcc': result.mcc, 'threshold': result.threshold, 'hyperparam': result.hyperparam}]), path)
        pre = variance_prefilter(ds.X[train], 5000)
        for k in K_GRID:
            for draw in range(N_RANDOM_DRAWS):
                path = checkpoint(output_root, f'random_r{repeat}_f{fold}_k{k}_d{draw}').with_suffix('.csv')
                if path.is_file():
                    continue
                rng = np.random.default_rng([C.dataset_seed_id(COHORT), seed, fold, k, draw])
                panel = rng.choice(pre, size=k, replace=False)
                with threadpool_limits(1):
                    result = evaluate_panel(ds.X[train][:, panel], ds.y[train], ds.X[test][:, panel], ds.y[test], classifier='lr_l2', seed=C.seed_for(C.dataset_seed_id(COHORT), seed, fold, k, draw), n_jobs=1)
                atomic_csv(pd.DataFrame([{'cohort': COHORT, 'repeat': repeat, 'fold': fold, 'method': 'random_k', 'k': k, 'draw': draw, 'classifier': 'lr_l2', 'auroc': result.auroc, 'bacc': result.bacc, 'mcc': result.mcc, 'threshold': result.threshold, 'hyperparam': result.hyperparam}]), path)
        print(f'completed reference checkpoints: {COHORT} r{repeat} f{fold}', flush=True)


In [ ]:
# Consolidate only this shard. Keep every shard archive for the final merge step.
files = sorted((output_root / 'checkpoints').glob('*.csv'))
rows = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)
b0 = rows[rows.method == 'b0'].sort_values(['repeat', 'fold'])
random_k = rows[rows.method == 'random_k'].sort_values(['repeat', 'fold', 'k', 'draw'])
atomic_csv(b0, output_root / 'b0_per_fold.csv')
atomic_csv(random_k, output_root / 'random_k_per_fold.csv')
config = {'experiment': 'E1_references_v3', 'cohort': COHORT, 'shard_name': SHARD_NAME, 'repeats': REPEATS, 'k_grid': K_GRID, 'run_b0': RUN_B0, 'random_k_draws': N_RANDOM_DRAWS, 'prefilter': 'top_5000_train_variance_unlabelled', 'classifier': 'lr_l2', 'split_sha256': sha256(split_file), 'commit_hash': COMMIT, 'traceable': True}
(output_root / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
archive = Path('/kaggle/working') / f'e1_references_{COHORT}_{SHARD_NAME}.zip'
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=WORK_ROOT, base_dir='results/survey_v3/E1_references')
print('DONE:', archive, 'rows:', len(rows))
